In [ ]:
import torch
import torch.nn as nn

from CvT import CvT
from dataset import CustomDataset

In [ ]:
DEVICE = torch.device("cuda")
DATA_DIR = "./data"

In [ ]:
from torchvision import transforms
from torch.utils.data import random_split

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = CustomDataset(root_dir=DATA_DIR, annotation_file=f"{DATA_DIR}/annotations/list.txt", transform=transform)

print(f'Dataset size: {len(dataset)}')
for i in range(5):
    image, label = dataset[i]
    print(f'Sample {i}: Image shape: {image.shape}, Label: {label}')
    # label = correct index

In [ ]:
from tqdm import tqdm
from tqdm import trange

In [ ]:
def train_fn(train_loader, model, optimizer, loss_fn, show_progress=False):
    loop = tqdm(train_loader, leave=True) if show_progress else train_loader
    losses = []

    for _, (img, label) in enumerate(loop):
        img = img.to(DEVICE)
        label = label.to(DEVICE)
        out = model(img)
        
        loss = loss_fn(out, label)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if show_progress:
            loop.set_postfix(loss=loss.item())

    mean_loss = sum(losses) / len(losses)
    return mean_loss

In [ ]:
def validate(loader, model, loss_fn, show_progress=False):
    model.eval()
    loop = tqdm(loader, leave=True) if show_progress else loader
    losses = []

    with torch.no_grad():
        for _, (img, label) in enumerate(loop):
            img = img.to(DEVICE)
            label = label.to(DEVICE)
            out = model(img)
            loss = loss_fn(out, label)
            losses.append(loss.item())

            if show_progress:
                loop.set_postfix(loss=loss.item())

    mean_loss = sum(losses) / len(losses)
    model.train()
    return mean_loss

In [ ]:
BATCH_SIZE = 32

WD = 1e-5
MOMENTUM = 0.9
LR = 3e-4

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [ ]:
model = CvT(batch_size=BATCH_SIZE, img_ch=3,
            attn_drop=0.1, proj_drop=0.1, mlp_drop=0.3,
            depth1=1, depth2=2, depth3=10,
            # Conv Embadding parameters
            k1=7, c1=64, s1=4, k2=3, c2=192, s2=2, k3=3, c3=384, s3=2,
            # Conv Proj parameters
            kp1=3, kp2=3, kp3=3,
            # MHSA parameters
            H1=1, H2=3, H3=6,
            # MLP parameters
            R1=4, R2=4, R3=4, num_classes=37)
model = model.to(DEVICE)

adam = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
sgd = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
optim = sgd
loss = nn.CrossEntropyLoss()

In [ ]:
EPOCHS = 300

In [ ]:
# 1. Overfit on small dataset
a = BATCH_SIZE * 10
b = BATCH_SIZE * 2
small_dataset, val_small_dataset, _ = random_split(dataset, [a, b, len(dataset) - (a+b)])
train_loader = torch.utils.data.DataLoader(small_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_small_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt
import os

train_losses = []
val_losses = []
best_val_loss = float('inf')
best_model_path = None

for epoch in trange(EPOCHS, desc="Training"):
    train_loss = train_fn(train_loader, model, optim, loss)
    val_loss = validate(val_loader, model, loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"{epoch + 1}_{round(val_loss, 4)}.pt")
        if best_model_path is not None and os.path.exists(best_model_path):
            os.remove(best_model_path)
        best_model_path = f"{epoch + 1}_{round(val_loss, 4)}.pt"

    # Live plot in Jupyter
    clear_output(wait=True)
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()